# 02 · Loop

The same model, given a second opinion. A judge reads the draft, says what is
missing, and the draft goes back with that attached.

> **You'll learn**
> - Register a second reasoner and call it from the first
> - Read a judge's verdict as structured data, not prose
> - Bound a loop so it cannot run forever

In [1]:
import os, sys, json, time, pathlib, httpx
sys.path.insert(0, str(pathlib.Path.cwd().parent / "lib"))
import dag

SERVER = os.environ.get("AGENTFIELD_SERVER", "http://localhost:8080")
GT = {g["id"]: g for g in json.load(open("../incidents/ground_truth.json"))["incidents"]}

def run(reasoner, **inp):
    """Dispatch to the node over HTTP and wait. Returns (run_id, output).

    Async on purpose: it hands back the run_id the DAG needs, and it does not
    die at the control plane's 90s synchronous ceiling.
    (Never `await app.call(...)` from a notebook — it executes children twice.)
    """
    r = httpx.post(f"{SERVER}/api/v1/execute/async/blast-radius.{reasoner}",
                   json={"input": inp}, timeout=30).json()
    eid, rid = r["execution_id"], r["run_id"]
    for _ in range(200):
        time.sleep(3)
        d = httpx.get(f"{SERVER}/api/v1/executions/{eid}", timeout=20).json()
        d = d.get("data", d)
        if d.get("status") in ("succeeded", "completed"):
            return rid, d.get("output") or d.get("result")
        if d.get("status") in ("failed", "error"):
            raise RuntimeError(d.get("error"))
    raise TimeoutError(eid)

def show(dx, title=""):
    if title:
        print(title); print("=" * len(title))
    print("root cause  :", dx["root_cause"])
    print("remediation :", dx["remediation"])
    print("confident   :", dx["confident"])
    for f in dx["findings"]:
        print(f"  - [{f['severity']:<8}] {f['location']}: {f['claim']}")


`r02` is two reasoners. `diagnose` drafts; `critique` judges. The loop is capped
at three drafts, always.

In [2]:
print(open("../node/rungs/r02.py").read())

"""r02 — loop.

One draft, then a judge. If the judge says the draft is thin, the draft goes back
in with the critique attached. Same schema out; a second opinion in the middle.
"""
from agentfield import AgentRouter
from pydantic import BaseModel, Field

from common import MODEL, NODE_ID, SYSTEM, Diagnosis, incident_text

router = AgentRouter(prefix="r02", tags=["rung", "r02"])

MAX_ROUNDS = 3


class Critique(BaseModel):
    """What the judge returns. `sufficient` is what stops the loop."""

    sufficient: bool = Field(description="True only if the diagnosis is settled by the cited evidence")
    what_is_missing: str = Field(description="One or two sentences: the specific gap the next draft must close")


@router.reasoner(tags=["judge"])
async def critique(incident_id: str, draft: dict) -> Critique:
    """Judge a draft against the incident. Harsh, specific, cheap."""
    return await router.app.ai(
        system=(
            "You review root-cause analyses. You are hard to satisf

## 1 · What the judge says about a one-shot answer

Take the answer chapter 01 produced for `inc-008` — memory pressure, raise the
limit — and hand it to the judge alone.

In [3]:
_, draft = run("r01_diagnose", incident_id="inc-008")
show(draft, "the draft being judged")

the draft being judged
root cause  : The notification-worker's render metrics hook (deployed in dep-2201) registers a listener on the shared EventEmitter for each render call, causing a memory leak of accumulated EventEmitter[render] instances that eventually exhausts the heap and triggers OOMKilled.
remediation : Immediately increase the memory limit to 3Gi to provide headroom, then fix the render metrics hook to register a single listener per emitter instead of one per render call, or remove the hook entirely. Also consider reducing the number of replicas or throttling the notification rate to reduce memory pressure.
confident   : True
  - [critical] heap_snapshot_summary: The heap snapshot shows 1,841,203 EventEmitter[render] instances retaining 812MB of memory, which is the direct evidence of a listener leak from the metrics hook.
  - [high    ] long_window: The daily max RSS per pod rose steadily from ~400MB before dep-2201 to 1394MB on July 14, indicating a memory leak that was t

In [4]:
_, verdict = run("r02_critique", incident_id="inc-008", draft=draft)
print("sufficient      :", verdict["sufficient"])
print("what is missing :", verdict["what_is_missing"])

sufficient      : True
what is missing : No missing check; the evidence conclusively identifies the listener leak from dep-2201 as the root cause.


That sentence is the whole mechanism. It is not a score, it is an instruction —
and it is what gets appended to the next draft's prompt.

## 2 · The loop, running

`diagnose` now does that automatically: draft, judge, redraft, up to three times.

In [5]:
rid, d = run("r02_diagnose", incident_id="inc-008")
show(d, "r02 on inc-008")

r02 on inc-008
root cause  : A memory leak introduced in deploy dep-2201 (version 5.2.0) where TemplateRenderer.ts registers a listener per render call on a module-level EventEmitter, and those listeners are never removed, accumulating 1.84 million listeners retaining ~812 MB. The leak was temporarily masked by a memory-limit increase (1 Gi to 1.5 Gi) on July 10 (dep-2244), but under July 15 peak load the accumulated RSS exceeded 1.5 Gi, causing repeated OOMKilled restarts.
remediation : Revert or fix the metrics hook in TemplateRenderer.ts to register global listeners only once, not per render call; temporarily increase memory limit to 2 Gi or higher to stop immediate restarts while a permanent fix is deployed.
confident   : True
  - [critical] metrics.json: Heap snapshot shows 1,841,203 EventEmitter[render] retainer objects holding 812 MB; this count equals the number of render calls since dep-2201, proving listeners accumulate and are never removed.
  - [critical] logs.txt: Log show

Ground truth again.

In [6]:
g = GT["inc-008"]
print("actual root cause:\n ", g["root_cause"]["summary"], "\n")
print("correct remediation:\n ", g["correct_remediation"]["immediate"])

actual root cause:
  Release 5.2.0 (dep-2201, nine days earlier) added a metrics hook that attaches a listener to a module-level EventEmitter on every render and never removes it. Each listener retains its RecipientContext, so RSS grows monotonically with cumulative messages processed rather than with concurrency. Daily peak RSS climbed from ~405MB to 1394MB over nine days and crossed the 1.5Gi limit during this morning's ordinary peak, producing the OOM kill loop. 

correct remediation:
  Roll back to 5.1.x, or ship a one-line fix that removes the listener after render (or uses a single module-level listener). A scheduled restart is an acceptable stopgap but must be labelled as one.


## 3 · The loop in the graph

Every judging round is a child execution. Counting them tells you how many drafts
it took before the judge was satisfied.

In [7]:
d_run = dag.fetch_run(rid, SERVER)
rounds = [e for e in d_run["executions"] if "critique" in (e.get("reasoner_id") or "")]
print(f"{len(rounds)} judging round(s)")
dag.render(rid, title="r02 · inc-008")

3 judging round(s)


**r02 · inc-008** — 4 executions · depth 2 · max fan-out 3 · 1 agent(s)

```mermaid
flowchart TD
  n0["r02_diagnose<br/><small>✓ succeeded · 123.3s</small>"]
  n1["r02_critique<br/><small>✓ succeeded · 9.3s</small>"]
  n2["r02_critique<br/><small>✓ succeeded · 44.4s</small>"]
  n3["r02_critique<br/><small>✓ succeeded · 23.6s</small>"]
  n0 --> n1
  n0 --> n2
  n0 --> n3
  class n0,n1,n2,n3 ok;
  classDef ok   fill:#dcfce7,stroke:#16a34a,stroke-width:1px,color:#14532d;
  classDef run  fill:#dbeafe,stroke:#2563eb,stroke-width:1px,color:#1e3a8a;
  classDef wait fill:#f1f5f9,stroke:#94a3b8,stroke-width:1px,color:#334155;
  classDef bad  fill:#fee2e2,stroke:#dc2626,stroke-width:1px,color:#7f1d1d;
```

## What you learned

- A reasoner can call another reasoner by name; the second one appears in the graph.
- A judge that returns `sufficient: bool` plus a gap turns critique into control flow.
- Caps are not optional — the loop stops at three drafts whether or not it agrees.

**Next:** `03_nested` — take the fixed plan away and let the model choose where to look.